# TFT Quick-Test (Colab/Kaggle)
Phase 2 smoke run. Verifies the version triangle + a 2-epoch fit on a data subset before committing the A6000 to a full run.
Bar to beat: LightGBM green val 2.18s / test 2.14s.

In [ ]:
# Pin the working triangle (record in MASTER_CONTEXT Section 3 once confirmed)
!pip -q install 'lightning==2.6.5' 'pytorch-forecasting==1.7.0' fastf1 pandera scipy
import torch; print('cuda', torch.cuda.is_available(), torch.__version__)
import pytorch_forecasting as pf, lightning; print('pf', pf.__version__, 'L', lightning.__version__)

In [ ]:
# Clone repo (token if private) + add to path
import os, sys
REPO='/content/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
sys.path.insert(0, REPO)
os.chdir(REPO)

In [ ]:
# Upload a few parquet files to data/raw/ (Colab: Files panel) OR re-ingest a subset.
# Need >=2 seasons for a real val split; for a smoke test 2022+2025 is enough.
import glob; print(sorted(glob.glob('data/raw/laps_*_r*.parquet'))[:5])

In [ ]:
from src.models.lap_time.train_tft import main
main(fast=True)   # 2 epochs, 5 train batches — just proves the pipeline runs end-to-end

If `fast=True` completes and logs VAL/TEST MAE without shape/encoder errors, the data contract + version triangle are good. Move to the A6000 and run `python -m src.models.lap_time.train_tft` for the full 100-epoch fit.

**A/B to run on the A6000, not here:** `min_encoder_length` 3 vs 5 (edit constant, two runs, compare val green MAE).